# Project Summary: AI Real Estate Consultant

This notebook demonstrates the construction of a hybrid **Retrieval-Augmented Generation (RAG)** system designed to compare real estate properties using both structured data and unstructured location descriptions.

### Key Components:

1.  **Data Preparation**:
    *   Loaded property listings from a cleaned CSV (`nawy_properties_cleaned.csv`).
    *   Extracted detailed location descriptions from a text file.

2.  **Vector Search Engine**:
    *   Utilized `RecursiveCharacterTextSplitter` to chunk location data.
    *   Generated embeddings using the `Qwen/Qwen3-Embedding-0.6B` model.
    *   Stored and indexed information in a **Chroma DB** vector store for semantic retrieval.

3.  **Hybrid RAG Pipeline**:
    *   **Property Retriever**: A custom function that fetches specific metadata (Price, Area, Beds) from a Pandas DataFrame based on unique IDs.
    *   **Location Retriever**: A LangChain-powered retriever that finds relevant geographical context from the vector database.
    *   **LLM Orchestration**: Integrated `ChatGroq` (Llama 3.1) to synthesize both data sources.

4.  **Comparison Logic**:
    *   Created a `compare_properties(id1, id2)` function that generates a comprehensive comparison report including markdown tables, location advantages, and investment recommendations.

### Results:
The system can now take any two property IDs and provide a deep-dive comparison that considers not just the price and size, but also the qualitative lifestyle and market benefits of their respective locations.

In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [19]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import pandas as pd
import json
from google.colab import userdata

In [4]:
# Retrieve the secret by the name you gave it
groq_api_key = userdata.get('GROQ_API_KEY')

# Now you can use it in your logic
print("API Key loaded successfully!")

API Key loaded successfully!


In [6]:
df = pd.read_csv("nawy_properties_cleaned.csv", dtype={"id": str})
df.head(1)

,id,location,property_name,property_type,description,m2,Beds,Baths,payment_plan,price,url_path,tag,cover_image,developer_logo,price_float,embedding_text
0,00000,New Capital City,"Apartment, Celia",Apartment,Apartment for sale in Celia - with 2 bedrooms ...,101.0,2.0,1.0,"32,083 EGP monthly / 2 Years","4,200,000 EGP",https://www.nawy.com/compound/623-celia/proper...,Resale,https://prod-images.cooingestate.com/processed...,https://prod-images.cooingestate.com/processed...,4200000.0,id: 00000 | location: New Capital City | prope...


In [7]:
loader = TextLoader("all_locations_descriptions.txt")
docs = loader.load()  # Returns list of Document objects

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Max characters per chunk
    chunk_overlap=200,   # Overlap for context
    add_start_index=True # Track original position
)
chunks = splitter.split_documents(docs)

In [ ]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

In [9]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./location_information_db"  # Saves locally
)

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})  # Top 10 chunks

# Fast Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=None)

# RAG prompt template
location_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context from location descriptions:
<context>{context}</context>

Question: {question}

Answer:"""
)

# Simple 2-step RAG chain: retrieve → format → prompt → llm → parse
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | location_prompt
    | llm
    | StrOutputParser()
)

In [11]:
# Usage
query = "Tell about this location 6th settlement"
response = rag_chain.invoke(query)
print(response)

6th Settlement is one of the elite neighborhoods in New Cairo City, Egypt. It is a recent upscale residential area, offering a premium location and many opulent compounds. The area is developed by the New Urban Communities Authority under the supervision of the Ministry of Housing, Utilities, and Urban Communities, and it works closely with New Cairo's Development Authority.

The location is situated at the endpoint of 5th Settlement, on the road between Ain Sokhna Road and Middle Ring Road, and is close to the New Administrative Capital. It is easily accessible from all major roads and is within close proximity to Nasr City, Heliopolis, Maadi, South 90 Street, and AUC.

6th Settlement is a well-planned neighborhood with a comprehensive living experience, featuring residential compounds, service zones, medical, administrative, and entertainment facilities, and mixed-use projects. The area is home to many upscale properties for sale, including lofts, studios, apartments, duplexes, penth

In [14]:
query = "which is a better location 6th settlement or Ras El Hekma?"
response = rag_chain.invoke(query)
print(response)

Based on the provided context, it's challenging to determine which location is better as it depends on individual preferences and priorities. However, I can provide some insights based on the information given.

6th Settlement is located in New Cairo, Egypt, and is described as having a prime location with easy access to major highways and destinations. It's close to the New Administrative Capital, Cairo International Airport, and other major cities. This location seems to be ideal for those who want to be close to the city and have easy access to various amenities.

Ras El Hekma, on the other hand, is located on the North Coast of Egypt and is described as a perfect summer gateway with pristine beaches, white sands, and a perfect climate. It's a popular destination for tourists and Egyptians alike, and is known for its upscale resorts and real estate projects.

Considering the two locations, Ras El Hekma seems to be a better option for those who:

* Want a beachfront location with acc

In [16]:
!zip -r location_information_db.zip location_information_db

  adding: location_information_db/ (stored 0%)
  adding: location_information_db/b978f195-09ec-4aa1-945a-8aa09f913531/ (stored 0%)
  adding: location_information_db/b978f195-09ec-4aa1-945a-8aa09f913531/header.bin (deflated 63%)
  adding: location_information_db/b978f195-09ec-4aa1-945a-8aa09f913531/data_level0.bin (deflated 100%)
  adding: location_information_db/b978f195-09ec-4aa1-945a-8aa09f913531/length.bin (deflated 63%)
  adding: location_information_db/b978f195-09ec-4aa1-945a-8aa09f913531/link_lists.bin (stored 0%)
  adding: location_information_db/chroma.sqlite3 (deflated 54%)


In [17]:
from google.colab import files
files.download("location_information_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
def get_properties_context(id1, id2, dataframe=df):
    """
    Takes two string IDs and returns property details as formatted text.
    """
    ids = [str(id1), str(id2)]
    selected_properties = dataframe[dataframe['id'].isin(ids)]

    context_parts = []
    for _, row in selected_properties.iterrows():
        prop_text = (
            f"Location: {row['location']}\n"
            f"Property Name: {row['property_name']}\n"
            f"Description: {row['description']}\n"
            f"Area: {row['m2']} m2\n"
            f"Beds: {row['Beds']}\n"
            f"Baths: {row['Baths']}\n"
            f"Payment Plan: {row['payment_plan']}\n"
            f"Price: {row['price']}"
        )
        context_parts.append(prop_text)

    return "\n\n---\n\n".join(context_parts)

# Example usage:
context = get_properties_context('00000', '00001')
print(context)

Location: New Capital City
Property Name: Apartment, Celia
Description: Apartment for sale in Celia - with 2 bedrooms in New Capital City by Talaat Moustafa Group (TMG) Holding.
Area: 101.0 m2
Beds: 2.0
Baths: 1.0
Payment Plan: 32,083 EGP monthly / 2 Years
Price: 4,200,000 EGP

---

Location: 6th settlement
Property Name: Apartment, Noi
Description: Apartment for sale in Noi - with 1 bedroom in 6th settlement by Urbnlanes Development.
Area: 81.0 m2
Beds: 1.0
Baths: 1.0
Payment Plan: 274,567 EGP quarterly / 8 Years
Price: 8,786,151 EGP


In [28]:
def property_retriever(inputs):
    """Retrieve property specs from dataframe"""

    id1 = inputs["id1"]
    id2 = inputs["id2"]

    return get_properties_context(id1, id2)


def location_retriever(inputs):
    """Retrieve location information from vectorstore"""

    query = f"locations of properties {inputs['id1']} and {inputs['id2']}"

    return vectorstore.as_retriever(
        search_kwargs={"k": 5}
    ).invoke(query)

def format_docs(docs):
    """Convert retrieved docs into text"""

    return "\n\n".join(doc.page_content for doc in docs)


multi_context_prompt = ChatPromptTemplate.from_template(
"""
You are a real estate expert.

Compare the following two properties and determine which one is better.

<Property_Details>
{property_context}
</Property_Details>

<Location_Context>
{location_context}
</Location_Context>

Provide a comparison based on just the context above including:

- Property specifications in markdown table format
- Location advantages
- Market trends
- Value for money

Finish with a clear final recommendation explaining which property is better and why.

Answer:
"""
)

# ✅ FIXED: Wrap functions with RunnableLambda
multi_context_chain = (
    {
        "property_context": RunnableLambda(property_retriever),  # ← Wrap here
        "location_context": (RunnableLambda(location_retriever) | format_docs)  # ← And here
    }
    | multi_context_prompt
    | llm
    | StrOutputParser()
)

def compare_properties(id1, id2):
    """
    Main function the user calls.
    Only requires two property IDs.
    """

    return multi_context_chain.invoke({
        "id1": id1,
        "id2": id2
    })


In [29]:
result = compare_properties("00000", "00001")

print(result)

Based on the provided information, I'll compare the two properties and provide a recommendation.

**Property Specifications**

| Property | Celia (New Capital City) | Noi (6th settlement) |
| --- | --- | --- |
| **Location** | New Capital City | 6th settlement |
| **Property Type** | Apartment | Apartment |
| **Area** | 101.0 m2 | 81.0 m2 |
| **Beds** | 2.0 | 1.0 |
| **Baths** | 1.0 | 1.0 |
| **Price** | 4,200,000 EGP | 8,786,151 EGP |
| **Payment Plan** | 32,083 EGP monthly / 2 Years | 274,567 EGP quarterly / 8 Years |

**Location Advantages**

* New Capital City (Celia):
	+ Strategic location with easy access to major highways and transportation hubs.
	+ Proximity to government institutions, embassies, and international organizations.
	+ Potential for long-term appreciation in property value.
* 6th settlement (Noi):
	+ Located in a growing area with new developments and infrastructure projects.
	+ Potential for increased property value due to urbanization and development.

**Market T